In [6]:
import torch
import torchvision
import matplotlib
import numpy
import scipy
import PIL
import skimage as ski
import pyvista #(for visulization)
import h5py

import os
import numpy as np
import h5py
from PIL import Image
import shutil

In [7]:
# Directorio donde están las imágenes .png
input_dir = 'docker_vm_test/01_traintest/czi_1000_ac_open_detector_br0%_patches/train'
dataset_name = "prueba"

In [8]:
# Crear una lista de archivos de imagen
image_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.png')])

In [9]:
# Leer las imágenes y apilarlas en un volumen 3D
volume = []
for file in image_files:
    img = Image.open(os.path.join(input_dir, file))
    img_array = np.array(img)
    volume.append(img_array)

volume = np.stack(volume, axis=-1)  # Crear el volumen 3D

ValueError: need at least one array to stack

In [ ]:
# Guardar el volumen como archivo h5
output_file = 'output_volume.h5'
with h5py.File(output_file, 'w') as f:
    f.create_dataset('volume', data=volume)

print(f'Volumen guardado en {output_file}')

In [ ]:
# Crear carpetas de destino
train_dir = f'data/{dataset_name}/train'
test_dir = 'data/test'
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

In [ ]:
# Mover archivos h5 a las carpetas correspondientes
for i, file in enumerate(os.listdir(input_dir)):
    src_path = os.path.join(input_dir, file)
    if i % 5 == 0:  # Usar el 20% para pruebas y el 80% para entrenamiento
        dest_path = os.path.join(test_dir, file)
    else:
        dest_path = os.path.join(train_dir, file)
    shutil.move(src_path, dest_path)

print('Archivos organizados en las carpetas train y test')

In [ ]:
# Crear un volumen de segmentación falso (por ejemplo, todas ceros)
segmentation_volume = np.zeros_like(volume)

In [ ]:
# Guardar el archivo de segmentación
segmentation_file = 'output_volume.seg.h5'
with h5py.File(segmentation_file, 'w') as f:
    f.create_dataset('volume', data=segmentation_volume)

print(f'Archivo de segmentación guardado en {segmentation_file}')

In [ ]:
# Renombrar y mover archivos según el formato esperado
shutil.copy(output_file, os.path.join(train_dir, 'volume1.im'))
shutil.copy(segmentation_file, os.path.join(train_dir, 'volume1.seg'))

In [ ]:
!python vox2vox/train.py --dataset "data/train" --n_epochs 2 --dataset_name "prueba" --img_height 512 --img_width 512 --img_depth 512

--n_epochs: number of epochs to train for

--glr: generator learning rate for Adam optimizer

--dlr: discriminator learning rate for Adam optimizer

--dataset_name: name of the dataset you are using, should be consistent with the name of the dataset folder in /data

--batch_size: training batch size, default is 1

--image_height/ --image_width/ --image_depth: image dimensions h/w/d

d_threshold: accuracy threshold in which the discriminator will be trained as long as its accuracy is below it